# Batch CV Parser

Process a directory of PDF/DOCX CVs with parallel LLM augmentation. Outputs combined CSV or JSON.

In [1]:
from pathlib import Path

# Project root: parent of this notebook's directory
PROJECT_PATH = Path.cwd().parent if (Path.cwd() / "batch_cv_parser.ipynb").exists() else Path.cwd()

# --- User configuration (paths relative to PROJECT_PATH) ---
CV_INPUT_DIR = PROJECT_PATH / "cv"                # Directory containing PDF/DOCX files
TEMP_DIR = PROJECT_PATH / "tmp"                   # Metadata JSON and augmented artifacts
OUTPUT_FILE = PROJECT_PATH / "output" / "combined.csv"  # Final output path
OUTPUT_FORMAT = "csv"                             # "csv" or "json"
MAX_WORKERS = 2                                   # Parallel LLM calls (2-4 recommended)
OPENAI_MODEL = "gpt-5.2"                           # None = use OPENAI_MODEL from .env
MAX_COMPLETION_TOKENS = 128000                     # Model max (e.g. 16384 for gpt-4o-mini)
CHUNK_SIZE = 80                                   # Max objects per LLM call; 0 = no chunking (single call)
DEDUPLICATE_ROWS = True                          # When True, merge duplicate title+year rows (keep published over in_progress)
VALIDATION_MAX_TRIES = 1                           # Max LLM revision attempts per chunk when validation fails


In [2]:
import os
from dotenv import load_dotenv
load_dotenv(PROJECT_PATH / ".env")
if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not set. Create a .env file with OPENAI_API_KEY=your-key")
print("API key loaded from .env")

API key loaded from .env


In [3]:
paths = sorted(CV_INPUT_DIR.glob("*.pdf")) + sorted(CV_INPUT_DIR.glob("*.docx")) + sorted(CV_INPUT_DIR.glob("*.doc"))
paths = [p for p in paths if p.suffix.lower() in (".pdf", ".docx", ".doc")]
if not paths:
    raise SystemExit(f"No PDF/DOCX files found in {CV_INPUT_DIR}")
print(f"Found {len(paths)} CV(s): {[p.name for p in paths]}")

Found 2 CV(s): ['deHaan_CV_25-26.pdf', "O'Reilly_CV_25-26.docx"]


## Extract lines and metadata (from extract_text.ipynb)

In [4]:
import re
from io import BytesIO

MIME = {".pdf": "application/pdf", ".docx": "application/vnd.openxmlformats-officedocument.wordprocessingml.document", ".doc": "application/msword"}

def _pdf_lines(doc):
    try:
        import pdfplumber
        with pdfplumber.open(BytesIO(doc)) as pdf:
            out = []
            for page in pdf.pages:
                text = page.extract_text(layout=True) or ""
                out.extend(text.splitlines())
            return out
    except ImportError:
        pass
    from pypdf import PdfReader
    reader = PdfReader(BytesIO(doc))
    out = []
    for page in reader.pages:
        try:
            text = page.extract_text(extraction_mode="layout") or ""
        except Exception:
            text = page.extract_text() or ""
        out.extend(text.splitlines())
    return out

def _docx_lines(doc):
    from docx import Document
    doc = Document(BytesIO(doc))
    out = []
    for p in doc.paragraphs:
        out.append(p.text)
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                for p in cell.paragraphs:
                    out.append(p.text)
    return out

def extract_lines(doc, mime, merge=True):
    if mime == "application/pdf":
        lines = _pdf_lines(doc)
    elif mime in ("application/vnd.openxmlformats-officedocument.wordprocessingml.document", "application/msword"):
        lines = _docx_lines(doc)
    else:
        raise ValueError(f"Unsupported: {mime}")
    return lines

In [5]:
def extract_line_metadata(document, mime, lines):
    from io import BytesIO
    numbered = [(i, ln.strip()) for i, ln in enumerate(lines, 1) if ln.strip()]
    if not numbered:
        return []
    def _format_hints():
        if mime == "application/pdf":
            return _pdf_format_hints(document, lines)
        if mime in ("application/vnd.openxmlformats-officedocument.wordprocessingml.document", "application/msword"):
            return _docx_format_hints(document, lines)
        return {}
    def _is_bold_font(fn):
        return fn and ("bold" in str(fn).lower() or "-b" in str(fn).lower() or "_b" in str(fn).lower())
    def _pdf_format_hints(doc, lns):
        try:
            import pdfplumber
        except ImportError:
            return {}
        plumber_lines, sizes = [], []
        with pdfplumber.open(BytesIO(doc)) as pdf:
            for page in pdf.pages:
                words = page.extract_words(extra_attrs=["size", "fontname"])
                if not words: continue
                rows = {}
                for w in words:
                    top = int(w.get("top", 0) or 0)
                    rows.setdefault(top, []).append(w)
                for top in sorted(rows):
                    ws = rows[top]
                    ws_sorted = sorted(ws, key=lambda w: (w.get("x0", 0), w.get("x1", 0)))
                    text = " ".join(w.get("text", "") or "" for w in ws_sorted).strip()
                    if not text: continue
                    szs = [float(w.get("size", 0) or 0) for w in ws if w.get("size") is not None]
                    max_sz = max(szs) if szs else 0
                    all_bold = bool(ws_sorted and all(_is_bold_font(w.get("fontname")) for w in ws_sorted))
                    if max_sz > 0: sizes.append(max_sz)
                    plumber_lines.append((text, max_sz, all_bold))
        if not sizes: return {}
        thresh = sorted(sizes)[len(sizes) // 2] * 1.1
        used = [False] * len(plumber_lines)
        numd = [(i, ln.strip()) for i, ln in enumerate(lns, 1) if ln.strip()]
        import unicodedata
        def norm(s): return unicodedata.normalize("NFKC", " ".join(s.split()))
        hints = {}
        for num, ln in numd:
            ln_n = norm(ln)
            for j, row in enumerate(plumber_lines):
                if not used[j] and norm(row[0]) == ln_n:
                    sz, bold = row[1], row[2]
                    hints[num] = (sz >= thresh and sz > 0) or bold
                    used[j] = True
                    break
            else:
                hints[num] = False
        return hints
    def _docx_format_hints(doc, lns):
        from docx import Document
        doc = Document(BytesIO(doc))
        hints, n = {}, 1
        for p in doc.paragraphs:
            style = (p.style and p.style.name or "").lower()
            runs = p.runs
            all_bold = bool(runs and all(getattr(r.font, "bold", None) for r in runs))
            hints[n] = "heading" in style or (all_bold and len(p.text.strip()) < 80)
            n += 1
        for t in doc.tables:
            for row in t.rows:
                for cell in row.cells:
                    for p in cell.paragraphs:
                        style = (p.style and p.style.name or "").lower()
                        runs = p.runs
                        all_bold = bool(runs and all(getattr(r.font, "bold", None) for r in runs))
                        hints[n] = "heading" in style or (all_bold and len(p.text.strip()) < 80)
                        n += 1
        return hints
    def _stem_line(line):
        from nltk.stem import SnowballStemmer
        stemmer = SnowballStemmer("english")
        words = re.findall(r"\b\w+\b", line.lower())
        return set(stemmer.stem(w) for w in words)
    def _categorize_header(line, kw_stems):
        line_stems = _stem_line(line)
        stemmed_str = " ".join(sorted(line_stems))
        best_cat, best_score = "other", 0
        for cat in ("publication", "presentation", "recognition"):
            score = len(line_stems & kw_stems[cat])
            if score > best_score: best_score, best_cat = score, cat
        best_cat = best_cat if best_score > 0 else "other"
        matched = line_stems & kw_stems[best_cat] if best_cat != "other" else set()
        return (best_cat, stemmed_str, matched)
    from nltk.stem import SnowballStemmer
    stemmer = SnowballStemmer("english")
    kw = {"publication": {"publication", "publications", "journal", "paper", "papers", "published", "article", "articles", "book", "books", "chapter", "chapters", "review", "working", "submitted", "accepted", "jae", "jar", "jf", "accounting", "round", "forthcoming", "revise", "revision", "conditionally"},
          "presentation": {"conference", "conferences", "workshop", "workshops", "keynote", "keynotes", "talk", "talks", "presentation", "presentations", "panel", "panels", "media", "interview", "interviews", "consortium", "faculty", "invited", "webinar", "podcast", "symposium"},
          "recognition": {"mentions", "mentioned", "recognition", "recognitions", "award", "awards", "honor", "honors", "fellowship", "fellowships", "grant", "grants", "prize", "prizes", "fellow", "professor", "chair", "selected", "distinguished", "outstanding", "accomplishment", "accomplishments", "appointment", "elected", "nominated"}}
    kw_stems = {cat: {stemmer.stem(w) for w in words} for cat, words in kw.items()}
    def _keyword_header_match(line, kw_stems):
        words = (line or "").strip().split()
        if not words or len(words) > 5: return False
        if not words[0] or not words[0][0].isupper(): return False
        line_stems = _stem_line(line)
        return any(line_stems & kw_stems[cat] for cat in ("publication", "presentation", "recognition"))
    format_hints = _format_hints()
    out = []
    for num, ln in numbered:
        fmt = format_hints.get(num, False)
        ml_cat, stemmed, matched = _categorize_header(ln, kw_stems)
        kw_match = _keyword_header_match(ln, kw_stems)
        is_header = fmt or kw_match
        obj = {"line_number": num, "line": ln, "is_header_candidate": is_header, "source": ["format"] if fmt else (["keyword"] if kw_match else ["none"]), "ml_category": ml_cat if is_header else "", "stemmed_words": stemmed}
        if is_header and ml_cat: obj["matched_stem"] = ",".join(sorted(matched)) if matched else ""
        out.append(obj)
    out = sorted(out, key=lambda x: x["line_number"])
    headers = [(m["line_number"], m["ml_category"], m.get("matched_stem", "")) for m in out if m["is_header_candidate"] and m["ml_category"]]
    for m in out:
        if not m["is_header_candidate"]:
            n = m["line_number"]
            prev = [(h[0], h[1], h[2]) for h in headers if h[0] < n]
            nearest = sorted(prev, key=lambda h: n - h[0])[:1]
            m["nearest_ml_category"] = nearest[0][1] if nearest else ""
            m["nearest_matched_stem"] = nearest[0][2] if nearest else ""
    return out

In [6]:
import json
import re
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Literal, Optional
from openai import OpenAI
from pydantic import BaseModel

class PublicationProposed(BaseModel):
    year: Optional[int] = None
    type: Literal["book", "journal", "other"]
    status: Literal["in_progress", "published"]
    institution: str = ""
    title: str = ""
    role: Literal["sole_author", "co_author"]

class PresentationProposed(BaseModel):
    title: str = ""
    year: Optional[int] = None
    type: Literal["conference", "keynote", "media", "workshop", "other"]
    role: Literal["sole_presenter", "co_presenter"]
    institution: str = ""

class RecognitionProposed(BaseModel):
    year: Optional[int] = None
    title: str = ""
    institution: str = ""
    references_work: Optional[str] = None

def _validate_proposed(m: dict) -> list:
    """Validate proposed object against asset_type. Returns list of error messages."""
    p = m.get("proposed")
    if not p: return []
    cat = m.get("ml_category") or m.get("nearest_ml_category") or "publication"
    errs = []
    if cat == "publication":
        pt = p.get("type")
        if pt not in ("book", "journal", "other"):
            if pt in (None, "", "null"):
                errs.append("publication type cannot be determined; reconsider if this line is correctly classified as publication (might be recognition, presentation, or other)")
            else:
                errs.append(f"publication type must be book|journal|other, got {pt}")
        if p.get("role") not in ("sole_author", "co_author", None, ""):
            errs.append(f"publication role must be sole_author|co_author, got {p.get('role')}")
    elif cat == "presentation":
        if p.get("type") not in ("conference", "keynote", "media", "workshop", "other", None, ""):
            errs.append(f"presentation type must be conference|keynote|media|workshop|other, got {p.get('type')}")
        if p.get("role") not in ("sole_presenter", "co_presenter", None, ""):
            errs.append(f"presentation role must be sole_presenter|co_presenter, got {p.get('role')}")
    elif cat == "recognition":
        if p.get("type") or p.get("role"):
            errs.append("recognition should not have type or role")
    return errs

def _clean_metadata(line_metadata):
    """Strip source/stemmed_words; keep all lines (no category filter)."""
    def _clean_obj(m):
        return {k: v for k, v in m.items() if k not in ("source", "stemmed_words")}
    return [_clean_obj(m) for m in line_metadata]

def _prev_incomplete(prev_line: str) -> bool:
    """True if prev line appears to continue (ends with comma, hyphen, or open paren)."""
    s = (prev_line or "").rstrip()
    return s.endswith((",", "-", "("))

def _curr_continues(curr_line: str) -> bool:
    """True if curr line appears to be a continuation (authors, venue, or line wrap)."""
    s = (curr_line or "").lstrip()
    if not s: return False
    if s.startswith(("•", "-", "–", "—")): return False  # bullet chars
    if re.match(r"^o\s{2,}", s): return False  # o  bullet variant
    if s.startswith("("): return True  # (with X, Y). Journal...
    if s[0].islower():
        if re.match(r"^[a-z][.)]\s", s): return False  # lettered bullet: a., b., c. or a), b), c)
        return True  # line wrap
    for prefix in ("Journal of", "Accounting Review", "Review of", "Management Science",
                   "Contemporary", "The Accounting", "The Journal"):
        if s.startswith(prefix): return True
    return False

def _build_merge_candidates(line_metadata: list) -> list:
    """Conservative merge: only when nearest_ml_category matches AND continuation signal (option 3)."""
    out = [dict(m) for m in line_metadata]
    for i in range(1, len(out)):
        curr, prev = out[i], out[i-1]
        if prev.get("nearest_ml_category") != curr.get("nearest_ml_category"): continue
        curr_ln = curr.get("line_number") or 0
        prev_ln = prev.get("line_number") or 0
        if curr_ln != prev_ln + 1: continue  # gap = empty line
        prev_line = prev.get("line") or ""
        curr_line = curr.get("line") or ""
        if not (_prev_incomplete(prev_line) or _curr_continues(curr_line)): continue
        curr["merge_into"] = prev_ln
    return out

def _apply_merge_candidates(metadata: list) -> list:
    """Collapse merge groups into single objects for LLM input. Returns reduced list. Tracks last_line_number for flagging."""
    merge_into_map = {m.get("line_number"): m.get("merge_into") for m in metadata if m.get("merge_into") is not None}
    def get_root(ln):
        while ln in merge_into_map:
            ln = merge_into_map[ln]
        return ln
    merged_by_first = {}
    for m in metadata:
        ln = m.get("line_number") or 0
        target = m.get("merge_into")
        if target is not None:
            root = get_root(target)
            merged_by_first[root]["line"] = (merged_by_first[root].get("line") or "") + " " + (m.get("line") or "")
            merged_by_first[root]["last_line_number"] = max(merged_by_first[root].get("last_line_number", root), ln)
        else:
            merged_by_first[ln] = dict(m)
            merged_by_first[ln]["last_line_number"] = ln
    return sorted(merged_by_first.values(), key=lambda x: x.get("line_number") or 0)

def _is_line_breaker(line: str) -> bool:
    """True if line starts with bullet/breaker (new item, not continuation)."""
    s = (line or "").lstrip()
    if not s: return True
    if s.startswith(("•", "-", "–", "—")): return True
    if re.match(r"^o\s{2,}", s): return True
    if re.match(r"^[a-z][.)]\s", s): return True  # a., b., a), b)
    if re.match(r"^\d+\.\s", s): return True  # 1., 2., 3.
    return False

def _curr_ends_complete(curr_line: str) -> bool:
    """True if curr ends with period (sentence/citation complete); merging next would produce period+no-space+char (abnormal)."""
    s = (curr_line or "").rstrip()
    return s.endswith(".") or s.endswith(".\"") or s.endswith(".'")

def _flag_merge_candidates(merged: list) -> list:
    """Flag consecutive lines (by line_number) with no breaker between them. Uses last_line_number when curr was merged from multiple lines."""
    for i in range(len(merged) - 1):
        curr, nxt = merged[i], merged[i + 1]
        curr_end = curr.get("last_line_number") or curr.get("line_number") or 0
        nxt_ln = nxt.get("line_number") or 0
        if nxt_ln != curr_end + 1: continue  # only consecutive in document order
        if nxt.get("is_header_candidate"): continue
        if _is_line_breaker(nxt.get("line") or ""): continue
        if _curr_ends_complete(curr.get("line") or ""): continue  # period at end = complete, next is new item
        curr["merge_candidate_with_next"] = True
    return merged

def _repair_truncated_json(s):
    idx = s.rfind("},")
    if idx < 0: idx = s.rfind("}")
    if idx >= 0:
        try:
            fixed = s[:idx + 1].rstrip().rstrip(",") + "]"
            return json.loads(fixed)
        except json.JSONDecodeError: pass
    return None

def parse_augmented(raw):
    t = raw.strip()
    if not t: raise RuntimeError("Empty LLM response")
    if "```json" in t:
        m = re.search(r"```json\s*([\s\S]*)", t)
        t = (m.group(1) or "").strip()
        if t.endswith("```"): t = t[:-3].strip()
    elif "```" in t:
        m = re.search(r"```\s*([\s\S]*?)```", t, re.DOTALL)
        t = m.group(1).strip() if m else t
    if t and not t.lstrip().startswith("["):
        m = re.search(r"\[[\s\S]*\]", t)
        t = m.group(0) if m else t
    try:
        augmented = json.loads(t)
    except json.JSONDecodeError as e:
        if "Unterminated" in str(e):
            augmented = _repair_truncated_json(t)
            if augmented is None: raise
        else: raise
    if isinstance(augmented, dict) and "lines" in augmented:
        augmented = augmented["lines"]
    if not isinstance(augmented, list):
        raise RuntimeError(f"Expected JSON array, got {type(augmented)}")
    return augmented

CSV_HEADERS = ["filename", "line_number", "asset_type", "year", "title", "asset_sub_type", "status", "role", "institution", "references_work"]

JOURNAL_MAP = {"JAE": "Journal of Accounting & Economics", "JAR": "Journal of Accounting Research", "JF": "Journal of Finance", "CAR": "Contemporary Accounting Research", "TAR": "The Accounting Review"}

def _fill_institution_from_line(institution, line):
    """When institution empty, infer from line using journal patterns."""
    if institution and str(institution).strip(): return institution
    line_upper = (line or "").upper()
    for abbr, full in JOURNAL_MAP.items():
        if abbr in line_upper or f" {abbr} " in f" {line_upper} " or line_upper.endswith(abbr):
            return full
    return institution or ""

def _proposed_to_row(m, p, filename):
    cat = m.get("ml_category") or m.get("nearest_ml_category") or "publication"
    inst = p.get("institution") or ""
    inst = _fill_institution_from_line(inst, m.get("line", ""))
    y = p.get("year")
    ref_work = (p.get("references_work") or "") if cat == "recognition" else ""
    return {"filename": filename, "line_number": m.get("line_number"), "asset_type": cat, "year": "" if y is None else str(y), "title": p.get("title") or "", "asset_sub_type": p.get("type") or "", "status": p.get("status") or "", "role": p.get("role") or "", "institution": inst, "references_work": ref_work}

def _deduplicate_rows(rows):
    """Merge rows with same filename+asset_type+title+year+institution; prefer published over in_progress."""
    by_key = {}
    for r in rows:
        k = (r["filename"], r.get("asset_type") or "", (r["title"] or "").strip(), str(r.get("year", "")).strip(), (r.get("institution") or "").strip())
        existing = by_key.get(k)
        if not existing or (r.get("status") == "published" and existing.get("status") != "published"):
            by_key[k] = r
    return list(by_key.values())

PROMPT_TEMPLATE = """AUGMENT the line metadata below. Each object has line_number, line, is_header_candidate, ml_category, nearest_ml_category, and optionally matched_stem (headers) or nearest_matched_stem (content under a header).

CRITICAL: Do NOT drop any lines. Output array MUST have exactly {n} objects, same order, same line_number values.

1) CATEGORIZATION: nearest_ml_category reflects document structure (the header above this line). Do NOT change it based on line content. A line under "Other activities, awards" is recognition even if it mentions presentations. Only correct when the header itself was misclassified. For content lines, leave ml_category empty; nearest_ml_category defines the category. Exception: if nearest_ml_category is 'other', infer a more suitable category from the line content and set ml_category to it.

2) MERGE LINES (only when needed): Only merge when lines were incorrectly broken up. When merging: update "line" on first line; for continuation lines add "merged_into": line_number and "proposed": null. Keep all objects.

2b) merge_candidate_with_next: When a line has merge_candidate_with_next: true, it means the next line has no bullet/breaker between them (potential line wrap). If citation extraction from a publication line is unsuccessful (incomplete title, missing journal, truncated authors), try merging with the next line's text and re-extract. Combine line text; update the first line's proposed; set next line's proposed: null and merged_into: line_number.

3) PROPOSED OBJECT: For each publication, presentation, or recognition item add "proposed" with the pydantic-shaped object. Use null for headers, pii, continuation lines.

Proposed shapes:
- publication: {{"year": 2020, "type": "book|journal|other", "status": "in_progress|published", "institution": "", "title": "", "role": "sole_author|co_author"}}
- presentation: {{"title": "", "year": 2020, "type": "conference|keynote|media|workshop|other", "role": "sole_presenter|co_presenter", "institution": ""}}
- recognition: {{"year": 2020, "title": "", "institution": "", "references_work": "optional title of paper/presentation when award/selection honors that work"}}

TYPE HINT: Use nearest_matched_stem when present to infer type. For publication: "book" (or "book,chapter") → type "book"; "journal"/"article" → "journal"; "chapter" alone → "other". Prefer "book" over "chapter" when both appear. For presentation: "conference"/"workshop" → "conference"; "keynote" → "keynote"; "media"/"interview"/"podcast" → "media".

Rules: institution = publisher for books, journal name for articles. role sole_author when CV owner only author, co_author when multiple or "with X". When author info is missing (e.g. working paper titles only), default to co_author.

Recognition: Include service roles (Managing Editor, Reviewer, Director, Editor, Faculty Member, Committee member) as recognition items with proposed objects. Use title for the role description, institution for the organization/journal. When a line under a publication section is a recognition (e.g., "Selected for X Conference", "Best Paper Award"), emit a recognition proposed object. Use references_work for the publication title from the preceding publication line when the recognition clearly refers to it.

REFERENCE - Journal abbreviations:
JAE = Journal of Accounting & Economics | JAR = Journal of Accounting Research | JF = Journal of Finance | CAR = Contemporary Accounting Research | TAR = The Accounting Review
Infer institution from line text: "Under review at JAE" → institution = "Journal of Accounting & Economics"; "Submitted to JF" → "Journal of Finance".

Years: "2024-25" → 2025; "this year"/"current" → use current year (2026); "forthcoming"/"in press" → null or most recent.

Example (publication):
Input: {{"line_number": 10, "line": "Smith, J. (2023). Title here. Journal of Accounting & Economics, 47(2), 123-145.", "ml_category": "publication"}}
Output: add "proposed": {{"year": 2023, "type": "journal", "status": "published", "institution": "Journal of Accounting & Economics", "title": "Title here", "role": "sole_author"}}

---
LINE METADATA:
{metadata_json}"""

REVISION_PROMPT_TEMPLATE = """REVISE the following proposed objects. They failed validation:
{errors_json}

Rules: publication → type in (book, journal, other), role in (sole_author, co_author). presentation → type in (conference, keynote, media, workshop, other), role in (sole_presenter, co_presenter). recognition → no type, no role; may have optional references_work (title of publication/presentation honored). "Selected for X Conference" is recognition, not publication. If category is wrong, include "ml_category" in the output object.

When publication type cannot be determined: Reconsider whether the line is correctly classified. It may be a recognition (award, service role, selection), a presentation, or not a publication. If so, change ml_category and provide the appropriate proposed object. If it is a publication but type is ambiguous, use "other" for working papers, chapters, or unclear cases.
Return JSON array of revised objects in same order. Each object: {{"proposed": {{...}}, "ml_category": "..."}} when category changes, or {{"proposed": {{...}}}} otherwise."""

def _augment_chunk(client, chunk, on_progress=None, chunk_idx=0, total_chunks=1):
    """Call LLM to augment a chunk of line metadata. Validate and retry until clean or max tries. Returns augmented list."""
    prompt = PROMPT_TEMPLATE.format(n=len(chunk), metadata_json=json.dumps(chunk, indent=2))
    response = client.chat.completions.create(
        model=OPENAI_MODEL or os.environ.get("OPENAI_MODEL", "gpt-5.2"),
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_completion_tokens=MAX_COMPLETION_TOKENS,
    )
    raw = response.choices[0].message.content or ""
    est_output = 2 * (len(json.dumps(chunk, indent=2)) // 4)
    if on_progress: on_progress(len(raw) // 4, est_output, chunk_idx, total_chunks)
    augmented = parse_augmented(raw)
    for attempt in range(VALIDATION_MAX_TRIES):
        bad = [(i, errs) for i, m in enumerate(augmented) if m.get("proposed") and (errs := _validate_proposed(m))]
        if not bad: break
        indices = [b[0] for b in bad]
        payload = [{"idx": i, "line_number": augmented[i].get("line_number"), "line": augmented[i].get("line"), "proposed": augmented[i]["proposed"], "errors": errs} for i, errs in bad]
        rev_prompt = REVISION_PROMPT_TEMPLATE.format(errors_json=json.dumps(payload, indent=2))
        rev_response = client.chat.completions.create(
            model=OPENAI_MODEL or os.environ.get("OPENAI_MODEL", "gpt-5.2"),
            messages=[{"role": "user", "content": rev_prompt}],
            temperature=0,
            max_completion_tokens=MAX_COMPLETION_TOKENS,
        )
        rev_raw = rev_response.choices[0].message.content or ""
        if on_progress: on_progress(len(rev_raw) // 4, 0, chunk_idx, total_chunks)
        revised = parse_augmented(rev_raw)
        for j, rev in enumerate(revised):
            if j >= len(indices): break
            idx = indices[j]
            obj = rev if isinstance(rev, dict) else {"proposed": rev}
            augmented[idx]["proposed"] = obj.get("proposed", obj)
            if "ml_category" in obj: augmented[idx]["ml_category"] = obj["ml_category"]
    return augmented

In [7]:
def process_one_cv(path, progress_entry=None, on_progress=None):
    """Process single CV: extract, metadata, clean, LLM augment. Returns list of CSV rows."""
    suffix = path.suffix.lower()
    mime = MIME.get(suffix)
    if not mime: return []
    document = path.read_bytes()
    lines = extract_lines(document, mime, merge=False)
    line_metadata = extract_line_metadata(document, mime, lines)
    if not line_metadata: return []
    cleaned = _clean_metadata(line_metadata)
    with_candidates = _build_merge_candidates(cleaned)
    merged = _apply_merge_candidates(with_candidates)
    merged = _flag_merge_candidates(merged)
    if progress_entry: progress_entry["total_lines"] = len(merged)
    TEMP_DIR.mkdir(parents=True, exist_ok=True)
    (TEMP_DIR / f"{path.stem}_line_metadata.json").write_text(json.dumps(line_metadata, indent=2), encoding="utf-8")
    (TEMP_DIR / f"{path.stem}_line_metadata_merge_candidates.json").write_text(json.dumps(with_candidates, indent=2), encoding="utf-8")
    (TEMP_DIR / f"{path.stem}_line_metadata_merged.json").write_text(json.dumps(merged, indent=2), encoding="utf-8")
    client = OpenAI()
    if CHUNK_SIZE and len(merged) > CHUNK_SIZE:
        chunks = [merged[i:i+CHUNK_SIZE] for i in range(0, len(merged), CHUNK_SIZE)]
        if progress_entry: progress_entry["total_chunks"] = len(chunks)
        workers = min(MAX_WORKERS, len(chunks))
        with ThreadPoolExecutor(max_workers=workers) as ex:
            futures = [ex.submit(_augment_chunk, client, ch, on_progress, i, len(chunks)) for i, ch in enumerate(chunks)]
            augmented_chunks = [f.result() for f in futures]
        augmented = [obj for ch in augmented_chunks for obj in ch]
    else:
        if progress_entry: progress_entry["total_chunks"] = 1
        augmented = _augment_chunk(client, merged, on_progress, 0, 1)
    aug_path = TEMP_DIR / f"{path.stem}_line_metadata_augmented.json"
    aug_path.write_text(json.dumps(augmented, indent=2), encoding="utf-8")
    rows = [_proposed_to_row(m, m["proposed"], path.name) for m in augmented if m.get("proposed")]
    return rows

In [8]:
import csv
PROGRESS_FILE = TEMP_DIR / "batch_progress.txt"
TEMP_DIR.mkdir(parents=True, exist_ok=True)
progress_data = {}
progress_lock = threading.Lock()

def write_progress():
    with progress_lock:
        lines = ["=== Batch CV Parser ===", ""]
        for name, d in sorted(progress_data.items()):
            s = d.get("streamed", 0)
            total_chunks = d.get("total_chunks", 1)
            chunks_done = min(d.get("chunks_done", 0), d.get("total_chunks", 1))
            tl = d.get("total_lines", 0)
            lp = tl if d.get("done") else min(int((chunks_done / max(1, total_chunks)) * tl), tl)
            lines_info = f"{lp}/{tl} lines" if tl else ""
            pct = min(99, 100 * lp // tl) if tl else 0
            if d.get("done"):
                status, extra = "done", f"{lines_info}  {s:,} tokens"
            elif pct > 0:
                status, extra = f"{pct}%", f"{lines_info}  {s:,} tokens"
            else:
                status, extra = "waiting", lines_info
            lines.append(f"  {name:<35} {status:>8}  {extra}")
        lines.append("")
        total_lines = sum(d.get('total_lines', 0) for d in progress_data.values())
        def _lp(d):
            if d.get('done'): return d.get('total_lines', 0)
            c = min(d.get('chunks_done', 0), d.get('total_chunks', 1))
            t, tl = max(1, d.get('total_chunks', 1)), d.get('total_lines', 0)
            return min(int(c * tl / t), tl) if tl else 0
        lines_processed = sum(_lp(d) for d in progress_data.values())
        done_count = sum(1 for d in progress_data.values() if d.get('done'))
        n_files = len(progress_data)
        if total_lines > 0:
            lines.append(f"  Total: {lines_processed}/{total_lines} lines ({done_count}/{n_files} files done)")
        else:
            lines.append(f"  Total: -- ({done_count}/{n_files} files done)")
        PROGRESS_FILE.write_text("\n".join(lines))

all_rows = []
n = len(paths)
print(f"Starting batch: {n} file(s). Live progress: tail -f {PROGRESS_FILE}")
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {}
    for path in paths:
        progress_data[path.name] = {"streamed": 0, "done": False, "total_lines": 0, "total_chunks": 1, "chunks_done": 0}
        def make_cb(p):
            def cb(streamed, est=0, chunk_idx=0, total_chunks=1):
                with progress_lock:
                    d = progress_data[p.name]
                    d["streamed"] = d.get("streamed", 0) + streamed
                    d["total_chunks"] = total_chunks
                    d["chunks_done"] = min(d.get("chunks_done", 0) + 1, total_chunks)
                write_progress()
            return cb
        futures[ex.submit(process_one_cv, path, progress_data[path.name], make_cb(path))] = path
    write_progress()
    for future in as_completed(futures):
        path = futures[future]
        with progress_lock:
            progress_data[path.name]["done"] = True
        try:
            rows = future.result()
            rows = sorted(rows, key=lambda r: (r.get("filename") or "", r.get("line_number") or 0))
            csv_path = TEMP_DIR / f"{path.stem}_output.csv"
            json_path = TEMP_DIR / f"{path.stem}_output.json"
            with csv_path.open("w", newline="", encoding="utf-8-sig") as f:
                w = csv.DictWriter(f, fieldnames=CSV_HEADERS)
                w.writeheader()
                w.writerows(rows)
            json_path.write_text(json.dumps(rows, indent=2), encoding="utf-8")
            all_rows.extend(rows)
        except Exception as e:
            pass  # progress already updated
        write_progress()
total_input_lines = sum(d.get('total_lines', 0) for d in progress_data.values())
PROGRESS_FILE.write_text(f"=== Batch CV Parser ===\n\n  Complete. Total: {len(all_rows)} rows from {total_input_lines} lines ({len(paths)} files)\n")
print(f"Total: {len(all_rows)} rows from {len(paths)} file(s)")

Starting batch: 2 file(s). Live progress: tail -f /Users/albert.wong/code/github/albertcwong/cv-parser2/tmp/batch_progress.txt
Total: 552 rows from 2 file(s)


In [9]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
rows_to_write = _deduplicate_rows(all_rows) if DEDUPLICATE_ROWS else all_rows
rows_to_write = sorted(rows_to_write, key=lambda r: (r.get("filename") or "", r.get("line_number") or 0))
from collections import Counter
by_cat = Counter((r.get("asset_type") or "no_category") for r in rows_to_write)
total_tokens = sum(d.get("streamed", 0) for d in progress_data.values())
summary = {"total": len(rows_to_write), "by_category": dict(by_cat), "total_tokens": total_tokens}
(TEMP_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
if OUTPUT_FORMAT == "csv":
    import csv
    with OUTPUT_FILE.open("w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=CSV_HEADERS)
        w.writeheader()
        w.writerows(rows_to_write)
    print(f"Wrote {len(rows_to_write)} rows to {OUTPUT_FILE}")
else:
    OUTPUT_FILE.write_text(json.dumps(rows_to_write, indent=2), encoding="utf-8")
    print(f"Wrote {len(rows_to_write)} rows to {OUTPUT_FILE}")

Wrote 551 rows to /Users/albert.wong/code/github/albertcwong/cv-parser2/output/combined.csv
